# 原39项中 SOXL 趋势与 SOXX 波动对现金选择的影响

冻结训练期标准化的两个既有决策前坐标及等权分数，对照 f696 原提案的成对排序。只读 Qlib 来源、完整账户区间；不拟合、不生成新资金账户。

In [1]:
import os
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
import hashlib,json
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.workflow import R
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr

root=Path.cwd();card=root/'research/stage1_soxl_cash_trend_risk_factor_attribution_20261024'
plan_path=card/'plan.json';plan=json.loads(plan_path.read_text());s=plan['sources']
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and all(plan['budget'][k]==0 for k in ('new_market_GETs','new_model_fits','new_native_accounts','new_bootstraps','new_Recorders'))
assert sha(root/s['factor_plan'])==s['factor_plan_sha256'] and sha(root/s['pairwise_plan'])==s['pairwise_plan_sha256']
f=root/s['factor_card'];a=root/s['labels_card'];p=root/s['f696_card'];b=root/s['account_card']
for file,key in [(f/'summary.json','factor_summary_sha256'),(f/'nas_readback.json','factor_nas_readback_sha256'),(a/'summary.json','labels_summary_sha256'),(p/'summary.json','f696_summary_sha256'),(b/'summary.json','account_summary_sha256')]:assert sha(file)==s[key],key
sf=json.loads((f/'summary.json').read_text());sn=json.loads((f/'nas_readback.json').read_text());sa=json.loads((a/'summary.json').read_text());sp=json.loads((p/'summary.json').read_text());sb=json.loads((b/'summary.json').read_text())
assert sf['nas_evidence']['sha256']==s['factor_nas_archive_sha256'] and sf['nas_evidence']['size_bytes']==s['factor_nas_size_bytes']
assert sn['NAS_archive_sha256']==s['factor_nas_archive_sha256'] and sn['artifact_hashes_checked']>=118
assert sa['nas_readback']['verified'] and sa['nas_readback']['sha256']==s['labels_nas_sha256']
assert sb['nas_readback']['verified'] and sp['nas_evidence']['sha256']==json.loads((p/'readback_verification.json').read_text())['archive_sha256']
fa=f/'qlib_recorder'/s['factor_experiment_id']/s['factor_recorder_id']/'artifacts'
aa=a/'qlib_recorder'/s['labels_experiment_id']/s['labels_recorder_id']/'artifacts'
pa=p/'qlib_recorder'/s['f696_experiment_id']/s['f696_recorder_id']/'artifacts'
ba=b/'qlib_recorder'/s['account_experiment_id']/s['account_recorder_id']/'artifacts'
assert sha(fa/'inputs/features')==s['feature_sha256'] and sha(fa/'inputs/timing')==s['timing_sha256']
assert sha(aa/'inputs/dataset')==s['labels_dataset_sha256'] and sha(pa/'inputs/frame')==s['f696_frame_sha256'] and sha(ba/'analysis/rolling126')==s['rolling_sha256']
assert sha(root/s['provider']/'qlib_store_manifest.json')==s['provider_manifest_sha256']
qlib.init(provider_uri=str((root/s['provider']).resolve()),region='us',expression_cache=None,dataset_cache=None,kernels=1)
with R.uri_context(str((f/'qlib_recorder').resolve())):
 rec=R.get_recorder(experiment_id=s['factor_experiment_id'],recorder_id=s['factor_recorder_id']);features=rec.load_object('inputs/features');timing=rec.load_object('inputs/timing')
with R.uri_context(str((a/'qlib_recorder').resolve())):
 rec=R.get_recorder(experiment_id=s['labels_experiment_id'],recorder_id=s['labels_recorder_id']);dataset=rec.load_object('inputs/dataset')
with R.uri_context(str((p/'qlib_recorder').resolve())):
 rec=R.get_recorder(experiment_id=s['f696_experiment_id'],recorder_id=s['f696_recorder_id']);frame=rec.load_object('inputs/frame').sort_values('trade_date').reset_index(drop=True)
with R.uri_context(str((b/'qlib_recorder').resolve())):
 rec=R.get_recorder(experiment_id=s['account_experiment_id'],recorder_id=s['account_recorder_id']);rolling=rec.load_object('analysis/rolling126')
original=sf['input_audit']['original_features'];assert len(original)==39 and list(features.columns[:39])==original
assert {'soxl_ret20','soxx_vol20'}.issubset(set(original)) and len(features)==len(timing)==3966 and len(frame)==944 and len(rolling)==819
calendar=pd.DatetimeIndex(D.calendar(start_time='2011-01-03',end_time='2026-10-09',freq='day'))
assert features.index.equals(calendar) and timing.index.equals(calendar)
tr=dataset.prepare('train',col_set=['label','meta']);he=dataset.prepare('held',col_set=['label','meta'])
tr=tr.loc[tr[('meta','train_mature')].astype(bool)];he=he.loc[he[('meta','held_mature')].astype(bool)]
assert (len(tr),len(he))==(3014,940)
tr_dates=pd.DatetimeIndex(tr.index.get_level_values('datetime'));he_dates=pd.DatetimeIndex(he.index.get_level_values('datetime'))
fdates=pd.DatetimeIndex(frame.signal_date);assert he_dates.equals(fdates[:940]) and tr_dates.isin(features.index).all()
decisions=pd.to_datetime(timing.loc[fdates,'decision_at'],utc=True).dt.tz_convert('America/New_York').dt.date
assert all(x>y.date() for x,y in zip(decisions,fdates))
base=features.loc['2011-01-03':'2022-12-30',['soxl_ret20','soxx_vol20']];assert len(base)==3020
med=base.median();std=base.std(ddof=0);assert np.isfinite(med).all() and np.isfinite(std).all() and (std>0).all()
zs=features[['soxl_ret20','soxx_vol20']].sub(med).div(std)
assert zs.loc[tr_dates].notna().all().all() and zs.loc[fdates].notna().all().all()
train_margin=tr[('label','SOXL_relative_H5')].to_numpy(float)-tr[('label','cash_advantage_H5')].to_numpy(float)
held_margin=he[('label','SOXL_relative_H5')].to_numpy(float)-he[('label','cash_advantage_H5')].to_numpy(float)
assert np.isfinite(train_margin).all() and np.isfinite(held_margin).all()
train_df=pd.DataFrame({'margin':train_margin,'trend':zs.loc[tr_dates,'soxl_ret20'].to_numpy(float),
                       'negative_vol':-zs.loc[tr_dates,'soxx_vol20'].to_numpy(float)},index=tr_dates)
train_df['primary']=train_df.trend+train_df.negative_vol
frame['margin']=np.r_[held_margin,[np.nan]*4]
frame['trend']=zs.loc[fdates,'soxl_ret20'].to_numpy(float)
frame['negative_vol']=-zs.loc[fdates,'soxx_vol20'].to_numpy(float)
frame['primary']=frame.trend+frame.negative_vol
frame['f696_intent']=frame.soxl_weight.to_numpy(float)-frame.cash_weight.to_numpy(float)
scores=['primary','trend','negative_vol','f696_intent']
def rank(g, names=scores):
 k=g.loc[np.isfinite(g.margin)];out={'sessions':len(g),'mature':len(k),'unknown':len(g)-len(k)}
 y=(k.margin.to_numpy(float)>1e-12).astype(int)
 out['positive']=int(y.sum());out['negative']=int(len(y)-y.sum())
 out['auc']={};out['spearman']={}
 for name in names:
  x=k[name].to_numpy(float)
  out['auc'][name]=float(roc_auc_score(y,x)) if len(set(y))==2 else None
  out['spearman'][name]=float(spearmanr(x,k.margin.to_numpy(float)).statistic) if len(k)>2 and len(set(x))>1 else None
 return out
train_rank=rank(train_df,scores[:3]);full=rank(frame)
annual=[{'year':int(year),'partial_endpoint':year in (2023,2026),**rank(g)} for year,g in frame.groupby(pd.to_datetime(frame.trade_date).dt.year,sort=True)]
quarter=[{'year':int(year),'quarter':int(q),**rank(g)} for (year,q),g in frame.groupby([pd.to_datetime(frame.trade_date).dt.year,pd.to_datetime(frame.trade_date).dt.quarter],sort=True)]
halves=[{'start':'2023-01-03','end':'2024-12-31',**rank(frame.iloc[:502])},{'start':'2025-01-02','end':'2026-10-07',**rank(frame.iloc[502:])}]
assert len(annual)==4 and len(quarter)==16 and halves[0]['mature']==502 and halves[1]['mature']==438
close=D.features(['SOXX'],['$close'],start_time='2011-01-03',end_time='2026-10-09',freq='day')['$close'].droplevel('instrument').sort_index()
assert close.index.equals(calendar)
windows=[]
for i in range(819):
 rw=rolling.iloc[i];g=frame.iloc[i:i+126];assert pd.Timestamp(g.trade_date.iloc[0])==pd.Timestamp(rw.first_trade)
 sig=pd.Timestamp(g.signal_date.iloc[0]);pos=calendar.get_loc(sig);assert pos>=20
 r5=float(close.iloc[pos]/close.iloc[pos-5]-1);r20=float(close.iloc[pos]/close.iloc[pos-20]-1)
 state='uptrend' if r5>=0 and r20>=0 else 'rebound' if r5>=0 else 'pullback' if r20>=0 else 'decline'
 windows.append({'year':int(pd.Timestamp(rw.first_trade).year),'start_state':state,'outcome':'weak' if rw.relative_log_excess<0 else 'good' if rw.relative_log_excess>0 else 'tie',**rank(g)})
w=pd.DataFrame(windows);assert w.outcome.value_counts().to_dict()=={'weak':564,'good':255} and w.mature.min()>=122
cells=[]
for (state,year),g in w.groupby(['start_state','year'],sort=True):
 weak=g[g.outcome.eq('weak')];good=g[g.outcome.eq('good')]
 row={'start_state':state,'year':int(year),'weak_windows':len(weak),'good_windows':len(good),'matched':bool(len(weak) and len(good))}
 if row['matched']:row['weak_minus_good_auc']={name:float(weak.auc.map(lambda d:d[name]).mean()-good.auc.map(lambda d:d[name]).mean()) for name in scores}
 cells.append(row)
matched=[x for x in cells if x['matched']];assert len(matched)==12 and all(x['good_windows']==0 for x in cells if x['year']==2023)
weights=np.array([min(x['weak_windows'],x['good_windows']) for x in matched],float)
matched_diff={name:float(np.average([x['weak_minus_good_auc'][name] for x in matched],weights=weights)) for name in scores}
rolling_summary={state:{'windows':len(g),'mean_auc':{name:float(g.auc.map(lambda d:d[name]).mean()) for name in scores},
                        'fraction_auc_above_half':{name:float((g.auc.map(lambda d:d[name])>.5).mean()) for name in scores}} for state,g in w.groupby('outcome')}
result={'status':'read_only_original39_trend_vol_SOXL_cash_factor_attribution_no_policy','plan_sha256':sha(plan_path),'source_recorder_ids':{'factor':s['factor_recorder_id'],'labels':s['labels_recorder_id'],'f696':s['f696_recorder_id'],'account':s['account_recorder_id']},
        'train_rank':train_rank,'held_full':full,'halves':halves,'annual':annual,'quarter':quarter,'rolling126_windows':len(w),'rolling_summary':rolling_summary,
        'matched_state_year_cells':len(matched),'matched_weak_minus_good_auc':matched_diff,'cells':cells,
        'feature_source_first_availability':'saved signal-day original39 inputs before next-session f696 09:29 ET decision','new_market_GETs':0,'new_model_fits':0,'new_native_accounts':0,'new_bootstraps':0,'new_Recorders':0,
        'independent_validation':False,'SOXX_goal_achieved':False,
        'limits':['Original 39 inputs and 2023-2026 H5 labels have been repeatedly reused; rank attribution is descriptive and not independent','H5 gross-price margin is not fee-net own-state account value','Overlapping rolling windows are not independent trials and cannot select a trade switch']}
(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
print({'train_auc':train_rank['auc'],'held_auc':full['auc'],'annual_auc':[(x['year'],x['auc']) for x in annual],'weak_good':rolling_summary})


{'train_auc': {'primary': 0.4861806689996059, 'trend': 0.48632765221935714, 'negative_vol': 0.49023266302721}, 'held_auc': {'primary': 0.4992296623931821, 'trend': 0.4966803769796288, 'negative_vol': 0.49662485715211035, 'f696_intent': 0.46799513276178756}, 'annual_auc': [(2023, {'primary': 0.427579493835172, 'trend': 0.43134328358208956, 'negative_vol': 0.5051914341336794, 'f696_intent': 0.3761194029850746}), (2024, {'primary': 0.4718034259318663, 'trend': 0.44928466029383457, 'negative_vol': 0.4734714826457946, 'f696_intent': 0.4870725604670559}), (2025, {'primary': 0.4980531686358754, 'trend': 0.5181928034371643, 'negative_vol': 0.493890977443609, 'f696_intent': 0.5007384532760473}), (2026, {'primary': 0.6107019562715765, 'trend': 0.5706559263521288, 'negative_vol': 0.5947065592635213, 'f696_intent': 0.5070195627157652})], 'weak_good': {'good': {'windows': 255, 'mean_auc': {'primary': 0.477897514716134, 'trend': 0.46582302556580574, 'negative_vol': 0.46358674457151017, 'f696_intent'

## 中文结论

# 已有绝对趋势和波动因子未修复 SOXL 对现金的排序

按冻结计划只读原39项 Qlib 输入、双H5标签、精确 f696 提案和原0bp Tiered账户全部819个连续126日窗口。固定候选为训练2011–2022期中位数/标准差标准化后的 `soxl_ret20 - soxx_vol20`，两个原坐标另作分量，方向与等权系数均在评价前固定；对照为原 `soxl_weight-cash_weight`。逐件校验来源 SHA、NAS 收据和限定读回、3020训练输入、3014成熟训练标签、940成熟评价标签、944提案及信号日早于下一交易日09:29决策。0新行情、拟合、账户、区块检验或Recorder。

固定组合在训练期预测“SOXL未来H5毛价格胜现金”的AUC **0.4862**，复用评价期 **0.4992**；完整2024/2025年分别 **0.4718/0.4981**，2023/2026两个部分年分别 **0.4276/0.6107**。前后固定半段 **0.4493/0.5461**。2026部分年较好不能抵消早期及两完整年失败。单独SOXL趋势和负SOXX波动的全段AUC分别 **0.4967/0.4966**；原 f696 意图为 **0.4680**。全部16季度及连续边际Spearman见 `result.json`。

原账户564个未胜SOXX滚动窗口中，组合平均AUC **0.4601**，255个胜出窗口为 **0.4779**；原 f696意图相应 **0.3671/0.4856**。事前SOXX 5/20日走势状态与窗口首年配对的12格，组合弱减好AUC **-0.0236**，原意图 **-0.0356**；2023四格仍无同格好窗口。819个窗口全部参与计算，但重叠窗口差异不是独立试验。

结论 `not_promoted_fixed_original39_trend_vol_pairwise_rank_failed`。这两个已有因子的直观等权组合虽减轻了原配置意图在弱窗口的反向排序，却没有稳定预测力，不进入0bp账户，也不按2026部分年改权重、翻符号或挑窗口救援。原配置错误不能简化为“加大20日SOXL动量、降低波动惩罚”。下一步应从**经济目标与接收方式**着手：在09:29事前状态上衡量现金转SOXL的自身风险容量与费用后边际价值，并与冻结f696动作链匹配；仍需单独冻结信息门及完整0bp含费自身状态账户。当前H5标签只是复用历史的毛价格结果，不是独立前向或SOXX净额超越证据。
